# 6장 — 안전한 확률 계산: log-sum-exp 🚑

웹 챕터의 두 구급법(max 빼기와 log 확률)을 코드로 구현하고,
"답은 한 자리도 안 바뀐다"(이동 불변성)를 assert 로 못박습니다.

> **노트북 사용법**: 각 셀을 클릭하고 `Shift + Enter`를 누르며 위에서 아래로 내려가세요.

**이 노트북의 목표**: LLM 이 훈련·평가 내내 쓰는 안전 회로.
**"max 를 빼고, 확률 대신 log 확률로 일한다."**

In [ ]:
# 🤖 AI 튜터 — 공부하다 막히면 오른쪽 아래 버튼을 눌러 물어보세요 (로컬 Ollama, 없어도 학습 지장 없음)
from tutor import chat
chat('ch6')

## 1. softmax 응급실: 순진한 계산 vs max 빼기

로짓 [89, 87, 84]. 5장에서 본 fp32 exp 의 사망 구간입니다.

In [ ]:
import numpy as np
f32 = np.float32

def naive_softmax(z):
    """교과서 공식 그대로 — exp 폭발 위험!"""
    e = np.exp(z)
    return e / e.sum()

def stable_softmax(z):
    """max 빼기 — 최대 로짓이 0 이 되어 exp ≤ 1, 폭발 불가."""
    e = np.exp(z - z.max())
    return e / e.sum()

z = np.array([89.0, 87.0, 84.0], dtype=np.float32)
with np.errstate(over='ignore', invalid='ignore'):
    p_naive = naive_softmax(z)
p_stable = stable_softmax(z)
print('순진한 softmax :', p_naive)
print('max 빼기 softmax:', np.round(p_stable, 4))

assert np.isnan(p_naive).any(), '순진한 계산: exp(89) = inf → inf/inf = NaN 붕괴 (5장의 변이)'
assert np.isclose(p_stable.sum(), 1.0, atol=1e-6), 'max 빼기: 멀쩡한 확률분포'

# 이동 불변성 — 로짓을 통째로 옮겨도 확률은 정확히 같다 (근사가 아니라 항등식)
z0 = np.array([0.0, -2.0, -5.0], dtype=np.float32)     # 차이가 같은 로짓
assert np.allclose(stable_softmax(z0), p_stable, atol=1e-6), 'softmax 는 차이만 본다'
print()
print('✔ 확인: 답은 그대로, 계산 경로만 안전한 쪽으로. 공짜 안전장치')

## 2. 문장 확률: 곱하기파의 전사 지점 실측

토큰 확률 p 로 n 토큰 문장의 확률 pⁿ. 그대로 곱하면 몇 토큰에서 죽을까요?

In [ ]:
def death_step(p, max_n=1000):
    """fp32 로 p 를 거듭 곱해 0 이 되는 걸음 수."""
    prob, n = f32(1.0), 0
    while prob > 0 and n < max_n:
        prob = f32(prob * f32(p))
        n += 1
    return n

for p in [0.5, 0.1, 0.01]:
    n = death_step(p)
    log10_at_death = n * np.log10(p)
    print(f'p = {p:<5} → {n:>3}토큰에서 언더플로 0   (그 시점 log₁₀ 확률 ≈ {log10_at_death:.0f})')

assert death_step(0.5) == 150, 'p=0.5: 150토큰 — 5장 ÷2 여행과 정확히 같은 걸음 수!'
assert death_step(0.1) in (45, 46), 'p=0.1: 한두 문장 분량이면 벌써 전사'
# log 파: 곱을 합으로 — 아무리 길어도 평범한 수
n_long = 10_000
log10_prob = n_long * np.log10(0.1)
assert log10_prob == -10_000 and np.isfinite(log10_prob), '만 토큰짜리 문장도 log 로는 −10000 — 그냥 수'
print()
print('✔ 확인: 확률 0 은 "불가능"이 아니라 칸이 못 담는 것. log 세계로 이사하면 해결')

## 3. log softmax 직행: 확률을 거치면 안 되는 이유

log(softmax(z)) 는 한 다리 건너 사고가 재발합니다. z − LSE(z) 직행과 비교.

In [ ]:
def logsumexp(z):
    """안정판 LSE = m + log Σ exp(z − m)."""
    m = z.max()
    return m + np.log(np.exp(z - m).sum())

z = np.array([0.0, -50.0, -200.0], dtype=np.float32)

with np.errstate(divide='ignore'):
    via_prob = np.log(stable_softmax(z))       # 확률을 만들고 log — 작은 쪽이 0 → log(0) = −inf!
direct = z - logsumexp(z)                       # log 세계 직행 — 전부 유한

print('확률 거쳐서 :', via_prob)
print('z − LSE 직행:', np.round(direct, 4))

assert np.isinf(via_prob).any(), 'exp(−200) 언더플로 → 확률 0 → log(0) = −inf 재발'
assert np.all(np.isfinite(direct)), '직행은 z = −200 이어도 그냥 −200 근처의 유한한 수'
assert np.isclose(float(logsumexp(z)), 0.0, atol=1e-6), '이 로짓의 LSE ≈ 0 (최대 로짓이 지배)'
print()
print('✔ 확인: PyTorch cross_entropy 가 내부에서 log_softmax 를 먼저 하는 이유')

## 4. 훈련 loss 의 정체: 채점은 처음부터 log 세계였다

5부 4장의 −log 채점, 6부 3장의 −log₂ q 채점판을 오늘의 도구로 다시 만들어 봅니다.

In [ ]:
# 미니 어휘 4개에 대한 로짓 — 정답은 2번 토큰이라고 하자
z = np.array([2.0, 0.5, 3.0, -1.0], dtype=np.float32)
answer = 2

log_probs = z - logsumexp(z)                    # 전 토큰의 log 확률 (자연로그)
loss = -log_probs[answer]                        # 크로스 엔트로피 = 정답의 −log p
bits = -log_probs[answer] / np.log(2)            # 6부 3장의 채점판은 −log₂ q (비트 단위)

print(f'정답 토큰의 확률   = {np.exp(log_probs[answer]):.4f}')
print(f'loss (−log p)      = {loss:.4f}')
print(f'채점판 (−log₂ q)   = {bits:.4f} 비트')

assert np.isclose(np.exp(log_probs).sum(), 1.0, atol=1e-5), 'log 확률을 되돌리면 합 1 — 멀쩡한 분포'
assert np.isclose(float(loss), float(-np.log(stable_softmax(z)[answer])), atol=1e-5), '같은 값 — 경로만 안전'
print()
print('✔ 확인: 훈련 로그의 loss = 정답 log 확률의 평균 × (−1). 여러분은 이미 log 세계 주민이었다')

## 5. 낭떠러지 그림: 곱하기파 vs log 파

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

p = 0.1
ns = np.arange(1, 61)
naive_vals = []
prob = f32(1.0)
for n in ns:
    prob = f32(prob * f32(p))
    naive_vals.append(float(prob))
naive_log10 = [np.log10(v) if v > 0 else None for v in naive_vals]
alive = [n for n, v in zip(ns, naive_vals) if v > 0]

plt.figure(figsize=(6.5, 3.8))
plt.plot(ns, ns * np.log10(p), '-', color='tab:green', label='log 파: n·log₁₀(0.1) (계속 직진)')
plt.plot(alive, [naive_log10[n - 1] for n in alive], 'o', ms=3, color='tab:purple', label='곱하기파 (fp32, 생존 구간)')
plt.axvline(max(alive) + 1, color='tab:red', ls='--', label=f'💀 {max(alive) + 1}토큰: 언더플로 낭떠러지')
plt.xlabel('문장 길이 n (토큰)'); plt.ylabel('log₁₀(문장 확률)'); plt.legend(fontsize=8); plt.grid(alpha=0.3)
plt.title('같은 값, 두 운명: 곱하기파는 낭떠러지, log 파는 직선'); plt.tight_layout(); plt.show()
print('✔ 확인: 5부의 퍼플렉시티·6부의 채점판이 전부 log 위에서 돌았던 이유')

**직접 해보기**: ① s1 의 로짓을 [1000, 998, 995] 로 바꿔도 max 빼기는 멀쩡한지 확인해 보세요.
② s2 의 `death_step(0.01)` 을 예측해 보고(힌트: 10⁻⁴⁵ 근처가 하한) 실행으로 확인해 보세요.

## 6. 관찰 정리 + 🤖 AI 연결

- softmax 를 그대로 계산하면 로짓 89 에서 💀. **max 빼기**(이동 불변성, exp(z−c) 약분)로
  답 불변 + 폭발 봉쇄. **LSE = m + log Σ exp(z−m)**, **log softmax = z − LSE** 직행.
- 확률의 곱은 언더플로行. **log 확률**(곱 → 합)로 이사: p=0.1 은 46토큰, p=0.5 는 150토큰
  (5장 ÷2 여행과 같은 수)에 곱하기파 전사.
- log(softmax) 는 log(0) = −inf 재발. 확률을 거치지 말 것(PyTorch cross_entropy 의 내부).
- **다시 쓰는 수학**: 4부 9장(softmax 이론 → 실전 구현체), 5부 4장(−log 채점 = log 세계),
  2부 7장(지수·로그 법칙), 5장(폭발·언더플로의 현장).

**다음 챕터**: 7장, 반쪽 자(fp16·bf16). 칸을 반으로 줄이는 유혹과,
지수·가수 비트를 어떻게 나눌 것인가의 갈림길 + 혼합 정밀도·loss scaling.